# DL_02 — Deep Learning Literature Review and Architecture Logic

**Owner: Aarya Srivastava**  
**Work package 2 of 5 — literature, architecture design and model validation**

## 1. Why Phase 2 needs a spatial model

Phase 1 maps a small vector of layout summaries to one scalar canopy-averaged wind-speed value. A scalar cannot identify where a wake occurs, how flow changes around individual buildings, or how a street canyon redistributes pedestrian-level velocity. Phase 2 therefore learns a mapping from a rasterized geometry plus simulation conditions to two spatial velocity-component fields.

The target is the pedestrian-level `uped` and `vped` arrays in the complete idealized UrbanTALES pedestrian NetCDF files. The supplied datasheet describes pedestrian-level quantities as sampled at 1.75 m. No temperature target is available in this idealized subset, so this project does not claim heat or temperature prediction.

## 2. Literature review

### U-Net

Ronneberger, Fischer and Brox (2015) introduced U-Net as an encoder-decoder convolutional architecture with a contracting path for context and a symmetric expanding path with skip connections for localization. Although the original application was biomedical segmentation, the same dense input-to-field structure is useful here: the output must preserve spatial alignment with the input layout. Skip connections allow fine building-edge and near-building patterns to reach the decoder, while deeper layers learn larger-scale interactions.

### Deep learning for physical fields

Farimani, Gomes and Pande (2017), *Deep Learning the Physics of Transport Phenomena*, studied conditional generative models for mapping domain/boundary information to steady heat-conduction and incompressible-flow solutions. This supports the broader idea that neural surrogates can map physical conditions to spatial solutions. It does not mean our model enforces the governing equations: our U-Net is supervised by UrbanTALES fields and must be described as a data-driven surrogate.

**Reference clarification:** the faculty brief names “Deep Learning for Wall Bounded Turbulent Flows” by Farimani et al. (2017). The readily verifiable 2017 Farimani paper is titled *Deep Learning the Physics of Transport Phenomena*. Confirm the exact faculty-supplied citation before final IEEE submission rather than silently treating the two titles as identical.

### Fluid-mechanics machine learning

Brunton, Noack and Koumoutsakos (2020) review machine learning methods in fluid mechanics and emphasize that model choice, physical knowledge and generalization matter. Our design follows that caution by keeping complete layouts/families out of training, reporting uncertainty/limitations, and avoiding claims that the surrogate replaces CFD.

## Verified references for the Phase 2 literature review

1. O. Ronneberger, P. Fischer, and T. Brox, “U-Net: Convolutional Networks for Biomedical Image Segmentation,” *MICCAI 2015*, Lecture Notes in Computer Science, vol. 9351, pp. 234–241, 2015, doi: 10.1007/978-3-319-24574-4_28.
2. A. B. Farimani, J. Gomes, and V. S. Pande, “Deep Learning the Physics of Transport Phenomena,” arXiv:1709.02432, 2017. The title differs from the wording in the faculty brief; confirm the intended faculty reference before final submission.
3. S. L. Brunton, B. R. Noack, and P. Koumoutsakos, “Machine Learning for Fluid Mechanics,” *Annual Review of Fluid Mechanics*, vol. 52, pp. 477–508, 2020, doi: 10.1146/annurev-fluid-010719-060214.
4. N. Nazarian et al., “UrbanTALES: A Large-Eddy Simulation Dataset for Urban Canopy Layer Turbulence and Parameterization,” *Bulletin of the American Meteorological Society*, vol. 106, no. 12, pp. E2461–E2478, 2025, doi: 10.1175/BAMS-D-25-0061.1.

## 3. Input/output design

The base U-Net receives five aligned channels:

1. normalized building-height raster;
2. sine of wind direction;
3. cosine of wind direction;
4. standardized friction velocity `u_tau`;
5. domain mask, which distinguishes the real grid from padding.

It returns two channels: `uped` and `vped`. The valid-air mask is **not** a prediction input. It is used only to calculate the masked loss and metrics, preventing building cells and missing CFD cells from affecting the objective.

## 4. Architecture logic

- **Encoder:** repeated 3×3 convolutions and downsampling enlarge the receptive field, allowing the network to learn layout-scale interactions.
- **Bottleneck:** combines broad spatial context.
- **Decoder:** transposed convolutions restore spatial resolution.
- **Skip connections:** concatenate encoder feature maps with decoder maps at matching scales, retaining local geometry information.
- **Two-channel output:** predicts both horizontal velocity components in one forward pass.
- **GroupNorm:** avoids dependence on large batch statistics because native grid sizes vary and the training loader uses one full layout per batch.
- **Dropout:** regularizes the bottleneck for the relatively small number of independent layouts.

The compact CNN is a simpler benchmark. It has same-resolution convolutions and dilated residual blocks but no U-Net encoder-decoder skip hierarchy. It helps test whether the U-Net structure adds value beyond a conventional convolutional field regressor.

## 5. Build the models and verify tensor dimensions

The largest spatial arrays are memory-intensive. This test uses a smaller dummy grid only to verify the architecture interface. The actual dataset loader preserves each case's native resolution and pads each case to a multiple of 16.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Locate the existing repository before importing the shared Phase 2 package.
current = Path.cwd().resolve()
candidates = [current, *current.parents]
ROOT = None
for candidate in candidates:
    if (candidate / "data" / "raw").is_dir() and (candidate / "Machine Learning Pipeline").is_dir():
        ROOT = candidate
        break
    if candidate.is_dir():
        for child in candidate.iterdir():
            if child.is_dir() and (child / "data" / "raw").is_dir() and (child / "Machine Learning Pipeline").is_dir():
                ROOT = child
                break
    if ROOT is not None:
        break
if ROOT is None:
    raise FileNotFoundError("Clone/open the existing T006 repository first; the Phase 2 folder must be added at its root.")

sys.path.insert(0, str(ROOT))
from dl_src.paths import get_project_paths
PATHS = get_project_paths(ROOT)
DL_RESULTS = PATHS["dl_results"]
FIELDS_DIR = PATHS["fields"]
FIGURES_DIR = PATHS["figures"]
MODELS_DIR = PATHS["models"]
PREDICTIONS_DIR = PATHS["predictions"]

print("Repository root:", ROOT)
print("Runtime outputs:", DL_RESULTS)


In [ ]:
import torch
from dl_src.models import UNetVelocity, CompactCNNVelocity, count_trainable_parameters

from dl_src.training import get_torch_device
DEVICE = get_torch_device()
print("Device:", DEVICE)

sample_input = torch.randn(1, 5, 128, 96, device=DEVICE)
unet = UNetVelocity(in_channels=5, out_channels=2, base_channels=8, dropout=0.10).to(DEVICE)
cnn = CompactCNNVelocity(in_channels=5, out_channels=2, hidden_channels=24, dropout=0.05).to(DEVICE)

with torch.no_grad():
    unet_output = unet(sample_input)
    cnn_output = cnn(sample_input)

print("Input tensor:", tuple(sample_input.shape))
print("U-Net output:", tuple(unet_output.shape))
print("CNN baseline output:", tuple(cnn_output.shape))
print("U-Net trainable parameters:", f"{count_trainable_parameters(unet):,}")
print("CNN trainable parameters:", f"{count_trainable_parameters(cnn):,}")

assert unet_output.shape == (1, 2, 128, 96)
assert cnn_output.shape == (1, 2, 128, 96)
print("PASS: both models preserve the spatial grid and return two components.")

## 6. Save a readable architecture diagram and summary

This diagram is generated by the notebook at runtime. It is not a pre-created result image.

In [ ]:
from pathlib import Path

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.set_xlim(0, 13)
ax.set_ylim(0, 5)
ax.axis("off")
blocks = [
    (0.2, 2.0, 1.5, 1.2, "5 input\nchannels"),
    (2.0, 2.0, 1.5, 1.2, "Encoder 1\n8 features"),
    (3.9, 2.0, 1.5, 1.2, "Encoder 2\n16 features"),
    (5.8, 2.0, 1.5, 1.2, "Bottleneck\n128 features"),
    (7.7, 2.0, 1.5, 1.2, "Decoder +\nskip paths"),
    (9.6, 2.0, 1.5, 1.2, "Decoder\n8 features"),
    (11.4, 2.0, 1.4, 1.2, "2 output\ncomponents"),
]
for x, y, w, h, label in blocks:
    rect = plt.Rectangle((x, y), w, h, fill=False, linewidth=1.8)
    ax.add_patch(rect)
    ax.text(x + w / 2, y + h / 2, label, ha="center", va="center", fontsize=9)
for i in range(len(blocks) - 1):
    x1 = blocks[i][0] + blocks[i][2]
    x2 = blocks[i + 1][0]
    ax.annotate("", xy=(x2, 2.6), xytext=(x1, 2.6), arrowprops={"arrowstyle": "->", "lw": 1.5})
ax.annotate("skip connections", xy=(8.2, 3.3), xytext=(3.0, 4.3), arrowprops={"arrowstyle": "->", "lw": 1.2}, ha="center")
ax.text(6.5, 0.8, "Masked field loss is calculated only on valid open-air cells", ha="center", fontsize=10)
ax.set_title("Compact U-Net for pedestrian-level velocity components", fontsize=14)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "unet_architecture_logic.png", dpi=170)
plt.show()
plt.close(fig)

architecture_summary = {
    "input_channels": [
        "building_height_normalized", "wind_direction_sin", "wind_direction_cos",
        "friction_velocity_standardized", "domain_mask"
    ],
    "output_channels": ["uped", "vped"],
    "base_channels": 8,
    "downsampling_stages": 4,
    "normalization": "GroupNorm",
    "activation": "GELU",
    "regularization": "Dropout2d in bottleneck; AdamW weight decay; early stopping",
    "loss": "masked mean squared error over valid air cells",
    "parameter_count_unet": count_trainable_parameters(unet),
    "parameter_count_compact_cnn": count_trainable_parameters(cnn),
}
with open(DL_RESULTS / "architecture_summary.json", "w", encoding="utf-8") as file:
    json.dump(architecture_summary, file, indent=2)
print(json.dumps(architecture_summary, indent=2))

## 7. Theoretical points to explain in the viva

- A convolution applies shared local filters, which is useful when the same local building/wake pattern may occur at different positions.
- Downsampling increases the receptive field, so deeper layers can use larger layout context.
- Skip connections preserve high-resolution features that could otherwise be lost during pooling.
- A 1×1 output convolution maps decoder features to two physical-field channels.
- The masked loss is necessary because a building cell is not an air-flow target location.
- The model is not a CFD solver and has no explicit Navier–Stokes residual or mass-conservation penalty. Do not claim it is physics-informed in that stronger sense.

## DL_02 completion checklist

- [ ] Literature sources and their limits are documented.
- [ ] Input/output channels are justified.
- [ ] U-Net and compact CNN are implemented.
- [ ] Both models pass a shape test.
- [ ] Parameter counts and architecture diagram are generated under `dl_results/`.

**Handoff:** Palak trains both models in DL_03.